In [ ]:
use database_name.schema_name

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, dayofweek, when, hour, month, dayname
)

session = get_active_session()

# --- Base table: orders ---
orders_df = (
    session.table("SILVER_ORDERS")
    .select(
        col("ORDER_ID"),
        col("customer_id"),
        col("restaurant_id"),
        col("STATE"),
        col("ORDER_TOTAL"),
        col("ORDER_TOTAL_MINUTES"),
        col("ORDER_TIME"),
        col("total_miles_travelled")
    )
)


# --- Customer ---
customer_df = (
    session.table("SILVER_CUSTOMER")
    .select(
        col("customer_id"),
        col("membership_type")
    )
)

# --- Restaurant ---
restaurant_df = (
    session.table("SILVER_RESTAURANT")
    .select(
        col("restaurant_id"),
        col("borough"),
        col("food_rating"),
        col("value_rating"),
        col("avg_experience_rating")
    )
)

# --- JOIN ---
fact_df = (
    orders_df
    .join(customer_df, "customer_id", "left")
    .join(restaurant_df, "restaurant_id", "left")
)

# --- Filter ---
fact_df = fact_df.filter(col("state") == "delivered")
fact_df = fact_df.filter(
    col("order_total").is_not_null() &
    col("order_time").is_not_null()
)

# --- Day of week ---
fact_df = fact_df.with_column(
    "day_of_week",
    dayofweek(col("order_time"))   # 0 = Sunday
)

# --- Time of day bucket ---
fact_df = fact_df.with_column(
    "time_of_day",
    when((hour(col("order_time")) >= 5) & (hour(col("order_time")) < 8), "Early Morning")
    .when((hour(col("order_time")) >= 8) & (hour(col("order_time")) < 12), "Morning")
    .when((hour(col("order_time")) >= 12) & (hour(col("order_time")) < 17), "Afternoon")
    .when((hour(col("order_time")) >= 17) & (hour(col("order_time")) < 21), "Evening")
    .when((hour(col("order_time")) >= 21) & (hour(col("order_time")) <= 23), "Night")
    .otherwise("Late Night")
)

# --- Add season ---
fact_df = fact_df.with_column(
    "season",
    when(month(col("order_time")).isin([12, 1, 2]), "Winter")
    .when(month(col("order_time")).isin([3, 4, 5]), "Spring")
    .when(month(col("order_time")).isin([6, 7, 8]), "Summer")
    .otherwise("Autumn")
)

# --- Save final table ---
fact_df.write.mode("overwrite").save_as_table("SalesForcastData")